# SWaT — Part 3: Real-Time Attack Detection (Updated)
**Changes:**
- `DEAD_COLS` trimmed to match final pipeline
- `FeatureExtractor` uses deviation + z-score features (NO CUSUM, NO absolute rolling mean)
- LIT sensor normalisation added (run-start baseline)
- `verbose=` removed from schedulers

**Architecture:**
```
PLC (192.168.5.194:1502)
  │  Modbus TCP FC3+FC1
  ▼
ModbusPoller → SlidingWindowBuffer
  │
  ▼
FeatureExtractor (deviation/zscore — run-agnostic)
  │
  ▼
EnsembleDetector (XGB 40% + MLP 35% + BiLSTM 25%)
  │
  ▼
AlertManager → log + CSV
```


## 0 · Imports & Load Models

In [ ]:
import time, threading, queue, logging, json as _json
from pathlib import Path
from collections import deque
from datetime import datetime

import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[logging.FileHandler("realtime_detection.log"),
              logging.StreamHandler()]
)
log = logging.getLogger("SWaT-RT")

MODELS_DIR = Path("saved_models")
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ATTACK_NAMES = {
    0:"Normal Operation",       1:"Tank Overflow Attack",
    2:"Chemical Depletion Attack", 3:"Membrane Damage Attack",
    4:"pH Manipulation Attack", 5:"Slow Ramp Attack",
    6:"Reconnaissance Scan",    7:"Denial of Service",
    8:"Replay Attack",          9:"Valve Manipulation Attack",
}
ALERT_LEVELS = {0:'INFO',1:'CRITICAL',2:'HIGH',3:'HIGH',
                4:'CRITICAL',5:'HIGH',6:'MEDIUM',7:'MEDIUM',8:'MEDIUM',9:'HIGH'}
NUM_CLASSES = 10
log.info(f"Device: {DEVICE}")


In [ ]:
bundle       = joblib.load(MODELS_DIR/"best_model_bundle.joblib")
SCALER       = bundle['scaler']
FEATURE_COLS = bundle['feature_cols']
DEAD_COLS    = bundle['dead_cols']
TEMPORAL_SENSORS = bundle.get('temporal_sensors',
    ['LIT_101','LIT_301','LIT_401','AIT_202','FIT_101',
     'FIT_401','Acid_Tank_Level','Chlorine_Residual','DPIT_301'])
WINDOW_SIZES = bundle.get('temporal_windows', [10, 25, 50])
FEATURE_TYPE = bundle.get('feature_type', 'deviation_zscore')

xgb_model = joblib.load(MODELS_DIR/"xgb_multiclass.joblib")
rf_model  = joblib.load(MODELS_DIR/"rf_multiclass.joblib")

log.info(f"Models loaded | Features: {len(FEATURE_COLS)} | Type: {FEATURE_TYPE}")


In [ ]:
class SWaTMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, num_classes, dropout=0.3):
        super().__init__()
        layers, prev = [], input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev,h),nn.BatchNorm1d(h),nn.ReLU(),nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, num_classes))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

class SWaTLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, num_classes, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True,
                            dropout=dropout if num_layers>1 else 0, bidirectional=True)
        D = 2
        self.attn = nn.Linear(hidden_dim*D, 1)
        self.norm = nn.LayerNorm(hidden_dim*D)
        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(hidden_dim*D, num_classes)
    def forward(self, x):
        out,_ = self.lstm(x)
        w   = torch.softmax(self.attn(out), dim=1)
        ctx = (w*out).sum(dim=1)
        return self.fc(self.drop(self.norm(ctx)))

INPUT_DIM = len(FEATURE_COLS)
SEQ_LEN   = 25

mlp_model = SWaTMLP(INPUT_DIM,[512,256,128,64],NUM_CLASSES).to(DEVICE)
lstm_model = SWaTLSTM(INPUT_DIM,128,2,NUM_CLASSES).to(DEVICE)

for name, model, pt in [('MLP',mlp_model,'mlp_multiclass_best.pt'),
                          ('LSTM',lstm_model,'lstm_best.pt')]:
    try:
        model.load_state_dict(torch.load(MODELS_DIR/pt, map_location=DEVICE))
        model.eval(); log.info(f"{name} weights loaded")
    except Exception as e:
        log.warning(f"{name} not loaded: {e}")


## 1 · Modbus Poller

In [ ]:
from pymodbus.client import ModbusTcpClient

HOLDING_REGS = {
    'FIT_101':0,'LIT_101':1,'MV_101':2,
    'AIT_201':3,'AIT_202':4,'AIT_203':5,'FIT_201':6,
    'Acid_Tank_Level':7,'Chlorine_Tank_Level':8,
    'Coagulant_Tank_Level':9,'Bisulfate_Tank_Level':10,
    'DPIT_301':12,'FIT_301':13,'LIT_301':14,
    'AIT_401':23,'AIT_402':24,'FIT_401':25,'LIT_401':26,
    'AIT_501':28,'FIT_501':31,'PIT_501':35,'PIT_502':36,'PIT_503':37,
    'TDS_Feed':41,'TDS_Permeate':42,'FIT_601':43,'Turbidity_Raw':44,
    'Chlorine_Residual':51,
}
COIL_MAP = {
    'P_101':0,'P_102':1,'P_203':2,'P_205':3,'P_206':4,
    'MV_201':5,'P_301':6,'MV_301':7,'MV_302':8,
    'P_401':9,'P_403':10,'P_501':11,
    'P_601':12,'P_602':13,'P_603':14,
    'Chemical_Low_Alarm':20,'High_Level_Alarm':21,'System_Run':23,
}

class ModbusPoller:
    def __init__(self, host='192.168.5.194', port=1502, unit_id=1, timeout=1.0, retries=3):
        self.host=host; self.port=port; self.unit_id=unit_id
        self.timeout=timeout; self.retries=retries
        self.client=None; self._lock=threading.Lock()

    def connect(self):
        self.client=ModbusTcpClient(self.host,port=self.port,timeout=self.timeout)
        ok=self.client.connect()
        if ok: log.info(f"Modbus connected: {self.host}:{self.port}")
        else: raise ConnectionError(f"Cannot connect to {self.host}:{self.port}")

    def disconnect(self):
        if self.client: self.client.close(); log.info("Modbus disconnected")

    def poll(self):
        for attempt in range(self.retries):
            try:
                result={}
                with self._lock:
                    rr=self.client.read_holding_registers(0,52,unit=self.unit_id)
                    if rr.isError(): raise IOError(f"FC3: {rr}")
                    regs=rr.registers
                    for name,addr in HOLDING_REGS.items():
                        if addr<len(regs):
                            val=regs[addr]
                            if name=='AIT_202': val=val/100.0  # PLC stores ×100
                            result[name]=float(val)
                    rc=self.client.read_coils(0,25,unit=self.unit_id)
                    if rc.isError(): raise IOError(f"FC1: {rc}")
                    for name,addr in COIL_MAP.items():
                        if addr<len(rc.bits): result[name]=int(rc.bits[addr])
                result['timestamp']=datetime.utcnow().isoformat()
                return result
            except Exception as e:
                log.warning(f"Poll {attempt+1}/{self.retries}: {e}")
                if attempt<self.retries-1: time.sleep(0.05)
        log.error("All poll attempts failed"); return None


## 2 · Feature Extractor (Deviation/Z-score — Run-Agnostic)

In [ ]:
class SlidingWindowBuffer:
    def __init__(self, maxlen=200):
        self._buf=deque(maxlen=maxlen); self._lock=threading.Lock()
    def push(self, reading):
        with self._lock: self._buf.append(reading)
    def get_window(self, n):
        with self._lock: buf=list(self._buf)
        return buf[-n:] if len(buf)>=n else buf
    def __len__(self):
        with self._lock: return len(self._buf)


class FeatureExtractor:
    """
    Produces the same features as training pipeline:
    - d_{s}_dt      : 1st derivative
    - {s}_dev{w}    : deviation from local rolling mean  (NOT absolute mean)
    - {s}_zscore{w} : rolling z-score
    - {s}_rstd{w}   : rolling std
    - {s}_norm      : for LIT sensors, normalised to run-start baseline
    - {col}_pct     : for tank level sensors, depletion fraction

    NO CUSUM — run-specific accumulation causes cross-run failure.
    """
    def __init__(self, feature_cols, temporal_sensors, window_sizes, scaler, dead_cols):
        self.feature_cols     = feature_cols
        self.temporal_sensors = temporal_sensors
        self.window_sizes     = window_sizes
        self.scaler           = scaler
        self.dead_cols        = set(dead_cols)
        # State per session
        self._history: deque = deque(maxlen=max(window_sizes)*2)
        self._run_starts: dict = {}   # {sensor: first_value}
        self._tank_starts: dict = {}
        self._n = 0

    def reset(self):
        self._history.clear()
        self._run_starts.clear()
        self._tank_starts.clear()
        self._n = 0

    def _rolling_stats(self, sensor: str, values: list, w: int):
        """Compute rolling mean and std over last w values."""
        window = values[-w:] if len(values)>=w else values
        arr    = np.array(window, dtype=float)
        mean   = arr.mean()
        std    = max(arr.std(), 0.001)
        return mean, std

    def extract(self, window: list):
        """window: list of raw poll dicts. Returns (1, n_features) scaled array or None."""
        if len(window) < 2: return None

        df_win = pd.DataFrame(window)

        # Drop dead columns
        df_win = df_win.drop(columns=[c for c in self.dead_cols if c in df_win.columns],
                             errors='ignore')

        # Bool → int
        for c in df_win.select_dtypes(include=bool).columns:
            df_win[c] = df_win[c].astype(np.int8)

        # Tank depletion normalisation
        TANK_COLS = ['Acid_Tank_Level','Chlorine_Tank_Level',
                     'Coagulant_Tank_Level','Bisulfate_Tank_Level']
        for col in TANK_COLS:
            if col not in df_win.columns: continue
            if col not in self._tank_starts:
                self._tank_starts[col] = float(df_win[col].iloc[0])
            start = self._tank_starts[col]
            df_win[f'{col}_pct'] = ((start - df_win[col]) / max(start,1)).clip(0,1)

        # Track rolling value history per sensor
        last_row = df_win.iloc[-1]
        current_vals = {}
        for s in self.temporal_sensors:
            if s in last_row: current_vals[s] = float(last_row[s])

        self._history.append(current_vals)
        self._n += 1

        # Derivative
        dt_series = df_win['elapsed_seconds'].diff().fillna(0.2).clip(lower=0.01)             if 'elapsed_seconds' in df_win.columns else pd.Series([0.2]*len(df_win))

        for s in self.temporal_sensors:
            if s not in df_win.columns: continue
            col_v = df_win[s].astype(float)
            dt    = dt_series

            # Derivative from last two rows
            df_win[f'd_{s}_dt'] = col_v.diff().fillna(0) / dt

            # LIT normalisation
            if s.startswith('LIT_'):
                if s not in self._run_starts:
                    self._run_starts[s] = float(col_v.iloc[0])
                run_base = self._run_starts[s]
                df_win[f'{s}_norm'] = (col_v - run_base) / max(run_base, 1)

            # Rolling stats on window (not global CUSUM)
            for w in self.window_sizes:
                roll_mean = col_v.rolling(w, min_periods=1).mean()
                roll_std  = col_v.rolling(w, min_periods=1).std().clip(lower=0.001).fillna(0.001)
                df_win[f'{s}_dev{w}']    = (col_v - roll_mean).fillna(0)
                df_win[f'{s}_zscore{w}'] = ((col_v - roll_mean) / roll_std).fillna(0)
                df_win[f'{s}_rstd{w}']   = roll_std

        # Align to training feature columns
        last = df_win.iloc[[-1]].copy()
        feat_vec = np.zeros((1, len(self.feature_cols)), dtype=np.float32)
        for i, col in enumerate(self.feature_cols):
            if col in last.columns:
                v = last[col].values[0]
                feat_vec[0, i] = 0.0 if (pd.isna(v) or np.isinf(float(v))) else float(v)

        return self.scaler.transform(feat_vec).astype(np.float32)


## 3 · Ensemble Detector

In [ ]:
class EnsembleDetector:
    def __init__(self, xgb_model, mlp_model, lstm_model,
                 seq_len=25, device=DEVICE, weights=(0.4,0.35,0.25)):
        self.xgb=xgb_model; self.mlp=mlp_model; self.lstm=lstm_model
        self.seq_len=seq_len; self.device=device
        self.w_xgb,self.w_mlp,self.w_lstm=weights
        self._seq_buf=deque(maxlen=seq_len)

    def _lstm_proba(self, feat_vec):
        self._seq_buf.append(feat_vec[0])
        if len(self._seq_buf)<self.seq_len: return None
        seq=np.array(self._seq_buf,dtype=np.float32)
        t=torch.FloatTensor(seq).unsqueeze(0).to(self.device)
        self.lstm.eval()
        with torch.no_grad():
            return torch.softmax(self.lstm(t),dim=1).cpu().numpy()[0]

    def predict(self, feat_vec):
        xgb_p = self.xgb.predict_proba(feat_vec)[0]
        mlp_p = torch.softmax(
            self.mlp(torch.FloatTensor(feat_vec).to(self.device)),dim=1
        ).cpu().detach().numpy()[0]
        lstm_p = self._lstm_proba(feat_vec)

        if lstm_p is not None:
            proba = self.w_xgb*xgb_p + self.w_mlp*mlp_p + self.w_lstm*lstm_p
        else:
            ws = self.w_xgb+self.w_mlp
            proba = (self.w_xgb/ws)*xgb_p + (self.w_mlp/ws)*mlp_p

        cid = int(np.argmax(proba))
        return {
            'class_id':   cid,
            'class_name': ATTACK_NAMES[cid],
            'confidence': float(proba[cid]),
            'probabilities': proba.tolist(),
            'model_votes': {
                'xgb':  int(np.argmax(xgb_p)),
                'mlp':  int(np.argmax(mlp_p)),
                'lstm': int(np.argmax(lstm_p)) if lstm_p is not None else None,
            },
            'is_attack':   cid>0,
            'alert_level': ALERT_LEVELS[cid],
        }

    def reset(self): self._seq_buf.clear()

detector  = EnsembleDetector(xgb_model, mlp_model, lstm_model, SEQ_LEN)
extractor = FeatureExtractor(FEATURE_COLS, TEMPORAL_SENSORS, WINDOW_SIZES, SCALER, DEAD_COLS)
log.info("Detector + extractor ready")


## 4 · Alert Manager

In [ ]:
class AlertManager:
    SUPPRESS_SECS = 10
    COLORS = {'CRITICAL':'[91m','HIGH':'[93m',
               'MEDIUM':'[94m','INFO':'[92m','RESET':'[0m'}
    ICONS  = {'CRITICAL':'[!!]','HIGH':'[!]','MEDIUM':'[i]','INFO':'[OK]'}

    def __init__(self, log_csv='alert_history.csv', conf_threshold=0.55):
        self.log_csv=Path(log_csv); self.conf_threshold=conf_threshold
        self._last={};  self._count=0; self._suppressed=0; self._history=[]
        if not self.log_csv.exists():
            self.log_csv.write_text("timestamp,class_id,class_name,confidence,alert_level
")

    def process(self, result, raw_reading):
        cid=result['class_id']; conf=result['confidence']; level=result['alert_level']
        ts=raw_reading.get('timestamp', datetime.utcnow().isoformat())
        if conf<self.conf_threshold and cid!=0:
            self._suppressed+=1; return
        now=time.time()
        if cid!=0:
            if now-self._last.get(cid,0)<self.SUPPRESS_SECS: return
            self._last[cid]=now; self._count+=1
        col=self.COLORS.get(level,''); rst=self.COLORS['RESET']
        icon=self.ICONS.get(level,'')
        print(f"{col}[{ts}] {icon} [{level}] {result['class_name']} (conf={conf:.3f}){rst}")
        if cid!=0: log.warning(f"ATTACK: {result['class_name']} conf={conf:.3f}")
        with open(self.log_csv,'a') as f:
            f.write(f"{ts},{cid},{result['class_name']},{conf:.4f},{level}
")
        self._history.append({**result,'timestamp':ts})

    def stats(self):
        print(f"
=== Alert Stats: fired={self._count}  suppressed={self._suppressed} ===")
        return self._history

alert_mgr = AlertManager(conf_threshold=0.55)


## 5 · Real-Time Detection Loop

In [ ]:
class RealTimeDetector:
    def __init__(self, poller, buffer, extractor, detector, alert_mgr,
                 poll_interval=0.2, window_size=50):
        self.poller=poller; self.buffer=buffer; self.extractor=extractor
        self.detector=detector; self.alert_mgr=alert_mgr
        self.poll_interval=poll_interval; self.window_size=window_size
        self._running=threading.Event(); self._q=queue.Queue(maxsize=100)
        self._stats={'polls':0,'errors':0,'preds':0,'attacks':0,'start':None}

    def _poll_thread(self):
        while self._running.is_set():
            t0=time.perf_counter()
            r=self.poller.poll()
            if r: self._q.put(r, block=False); self._stats['polls']+=1
            else: self._stats['errors']+=1
            time.sleep(max(0,self.poll_interval-(time.perf_counter()-t0)))

    def run(self, duration_s=None, verbose_every=50):
        self._running.set(); self._stats['start']=time.time()
        self.extractor.reset(); self.detector.reset()
        threading.Thread(target=self._poll_thread, daemon=True).start()
        log.info(f"Detection started | poll={self.poll_interval}s window={self.window_size}")
        try:
            while self._running.is_set():
                if duration_s and (time.time()-self._stats['start'])>=duration_s: break
                try: reading=self._q.get(timeout=1.0)
                except queue.Empty: continue
                self.buffer.push(reading)
                if len(self.buffer)<max(self.window_size//2,10): continue
                feat=self.extractor.extract(self.buffer.get_window(self.window_size))
                if feat is None: continue
                result=self.detector.predict(feat)
                self._stats['preds']+=1
                if result['is_attack']: self._stats['attacks']+=1
                self.alert_mgr.process(result, reading)
                if self._stats['preds']%verbose_every==0:
                    el=time.time()-self._stats['start']
                    log.info(f"[{el:.0f}s] polls={self._stats['polls']} "
                             f"preds={self._stats['preds']} attacks={self._stats['attacks']}")
        except KeyboardInterrupt: log.info("Stopped by user")
        finally: self.stop()

    def stop(self):
        self._running.clear(); self.alert_mgr.stats()
        log.info(f"Final: {self._stats}")


## 6 · Run Live (PLC Connected)

In [ ]:
PLC_HOST='192.168.5.194'; PLC_PORT=1502
poller=ModbusPoller(host=PLC_HOST, port=PLC_PORT)
buffer=SlidingWindowBuffer(maxlen=200)
rt=RealTimeDetector(poller,buffer,extractor,detector,alert_mgr,poll_interval=0.2,window_size=50)
try:
    poller.connect()
    print("PLC connected — running (Ctrl+C to stop)")
    rt.run()
except ConnectionError as e:
    print(f"PLC not reachable: {e}
Use simulation cell below.")
finally:
    poller.disconnect()


## 7 · Simulation Mode (CSV Replay)

In [ ]:
ID_REMAP = {0:0,8:1,9:2,10:3,11:4,12:5,13:6,14:7,15:8,16:9}

class CSVReplayPoller:
    def __init__(self, csv_path, speed=20.0, loop=False):
        df=pd.read_csv(csv_path, compression='gzip' if str(csv_path).endswith('.gz') else None)
        df['ATTACK_ID']=df['ATTACK_ID'].map(ID_REMAP)
        self._rows=df.to_dict('records'); self._idx=0
        self._speed=speed; self._loop=loop; self._gt=[]
    def connect(self): log.info(f"CSV replay: {len(self._rows):,} rows")
    def disconnect(self): log.info("Replay done")
    def poll(self):
        if self._idx>=len(self._rows):
            if self._loop: self._idx=0
            else: return None
        r=self._rows[self._idx].copy()
        self._gt.append(int(r.get('ATTACK_ID',0)))
        r['timestamp']=datetime.utcnow().isoformat()
        self._idx+=1; time.sleep(0.2/self._speed); return r
    @property
    def ground_truth(self): return self._gt

csv_p   = CSVReplayPoller("compressed_data_csv.gz", speed=20.0)
sim_buf = SlidingWindowBuffer(maxlen=200)
sim_ext = FeatureExtractor(FEATURE_COLS,TEMPORAL_SENSORS,WINDOW_SIZES,SCALER,DEAD_COLS)
sim_det = EnsembleDetector(xgb_model,mlp_model,lstm_model,SEQ_LEN)
sim_alrt= AlertManager(log_csv='sim_alerts.csv', conf_threshold=0.55)
sim_rt  = RealTimeDetector(csv_p,sim_buf,sim_ext,sim_det,sim_alrt,poll_interval=0.0,window_size=50)

csv_p.connect()
print("Simulation running (60s window at 20x speed)...")
sim_rt.run(duration_s=60, verbose_every=100)
csv_p.disconnect()


## 8 · Simulation Results

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

PALETTE=['#1565C0','#E53935','#2E7D32','#F57F17','#6A1B9A',
         '#00838F','#AD1457','#37474F','#558B2F','#4E342E']
ATTACK_NAMES_LIST=[ATTACK_NAMES[i] for i in range(NUM_CLASSES)]

history=sim_alrt._history
if not history:
    print("No predictions yet — run simulation cell first"); 
else:
    pred_ids =[h['class_id'] for h in history]
    confs    =[h['confidence'] for h in history]
    gt=csv_p.ground_truth
    offset=len(gt)-len(pred_ids)
    gt_al=gt[max(0,offset):offset+len(pred_ids)]
    n=min(len(pred_ids),len(gt_al))
    pred_ids=pred_ids[:n]; gt_al=gt_al[:n]; confs=confs[:n]

    print("=== Simulation Report ===")
    print(classification_report(gt_al,pred_ids,target_names=ATTACK_NAMES_LIST,
                                 labels=list(range(NUM_CLASSES)),digits=4,zero_division=0))

    fig,axes=plt.subplots(2,2,figsize=(16,10))

    # Timeline
    ax=axes[0,0]
    ax.step(range(n),gt_al,where='post',color='#212121',lw=1,label='Truth',alpha=0.7)
    ax.step(range(n),pred_ids,where='post',color='#E53935',lw=1,ls='--',label='Pred',alpha=0.7)
    ax.set_title("Ground Truth vs Predicted",fontweight='bold')
    ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels([f"[{i}]" for i in range(10)],fontsize=7)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    # Confidence
    ax=axes[0,1]
    atk_m=np.array(pred_ids)>0
    ax.scatter(np.where(~atk_m)[0],np.array(confs)[~atk_m],s=3,color='#1565C0',alpha=0.3,label='Normal')
    ax.scatter(np.where(atk_m)[0], np.array(confs)[atk_m], s=5,color='#E53935',alpha=0.5,label='Attack')
    ax.axhline(0.55,color='orange',ls='--',lw=1.2,label='Threshold')
    ax.set_title("Detection Confidence",fontweight='bold'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

    # Confusion matrix
    ax=axes[1,0]
    cm=confusion_matrix(gt_al,pred_ids,normalize='true',labels=list(range(NUM_CLASSES)))
    sns.heatmap(cm,annot=True,fmt='.2f',cmap='Blues',ax=ax,
                xticklabels=[f"[{i}]" for i in range(10)],
                yticklabels=[f"[{i}]" for i in range(10)],
                linewidths=0.4,annot_kws={'size':7})
    ax.set_title("Simulation CM",fontweight='bold'); ax.tick_params(labelsize=7)

    # Per-class recall
    ax=axes[1,1]
    diag=cm.diagonal()
    ax.barh([ATTACK_NAMES[i][:16] for i in range(NUM_CLASSES)][::-1],diag[::-1],
            color=PALETTE[::-1])
    ax.axvline(0.8,color='red',ls='--',lw=1.2,label='80% target')
    ax.set_xlim(0,1.1); ax.set_title("Per-class Recall",fontweight='bold')
    ax.legend(fontsize=8); ax.grid(axis='x',alpha=0.3)

    plt.suptitle("Real-Time Simulation Results",fontweight='bold',fontsize=14)
    plt.tight_layout()
    plt.savefig("plots/15_simulation_results.png",dpi=150,bbox_inches='tight'); plt.show()


## 9 · Quick Reference

### Tuning parameters
| Parameter | Where | Default | Effect |
|-----------|-------|---------|--------|
| `conf_threshold` | AlertManager | 0.55 | Raise → fewer false alarms |
| `SUPPRESS_SECS` | AlertManager | 10 | Repeat-alert cooldown |
| `weights` | EnsembleDetector | (0.4,0.35,0.25) | XGB/MLP/LSTM blend |
| `window_size` | RealTimeDetector | 50 | Feature window (10s at 5Hz) |
| `SEQ_LEN` | Global | 25 | LSTM sequence window (5s) |
| `poll_interval` | RealTimeDetector | 0.2 | Seconds between polls |

### Feature type
This version uses **deviation + z-score features** (run-agnostic).  
No CUSUM, no absolute rolling means — ensures cross-run generalisation.

### Deploy
```python
bundle = joblib.load("saved_models/best_model_bundle.joblib")
# bundle['feature_type'] == 'deviation_zscore'
X_scaled = bundle['scaler'].transform(X_new)
preds    = bundle['model'].predict(X_scaled)
```
